# Análise Exploratória das Avaliações da Steam

Este notebook apresenta as etapas de preparação, exploração e análise inicial
das avaliações de usuários da plataforma Steam. A base será posteriormente
utilizada nas etapas de Processamento de Linguagem Natural (NLP) e classificação
automática das avaliações.

## 1. Importação das bibliotecas

Nesta etapa são carregadas as bibliotecas necessárias para leitura dos dados,
manipulação das informações e realização das análises exploratórias.

In [23]:
import gzip
import json
from pathlib import Path
import pandas as pd

## 2. Definição dos arquivos utilizados

Nesta etapa são definidos os caminhos dos arquivos que serão utilizados no
projeto. O trabalho será desenvolvido exclusivamente com a base de avaliações
da Steam, não sendo utilizado o dataset adicional de jogos.

In [24]:
PASTA_PROJETO = Path.cwd().parent  # Sobe um nível a partir da pasta 'notebooks'

ARQUIVO_REVIEWS = (
    PASTA_PROJETO
    / "Data"
    / "steam_2025_5k-dataset-reviews_20250901.json.gz"
)

print("Arquivo de reviews:", ARQUIVO_REVIEWS)
print("Arquivo existe?", ARQUIVO_REVIEWS.exists())

Arquivo de reviews: c:\Users\Francisco PC\OneDrive\Documentos\GitHub\projeto-aplicado-II-classificacoes-avaliacoes\Data\steam_2025_5k-dataset-reviews_20250901.json.gz
Arquivo existe? True


## 3. Carregamento das avaliações

O arquivo de avaliações está armazenado em formato JSON compactado (`.json.gz`).
Nesta etapa, os dados são carregados em memória para posterior transformação
e análise.

In [25]:
# Carregar o dataset de avaliações
with gzip.open(ARQUIVO_REVIEWS, "rt", encoding="utf-8") as arquivo:
    dados_reviews = json.load(arquivo)

print("\nArquivo de reviews carregado com sucesso!")
print("Chaves principais:", list(dados_reviews.keys()))


Arquivo de reviews carregado com sucesso!
Chaves principais: ['metadata', 'reviews']


## 4. Estruturação dos dados

As avaliações carregadas são transformadas em um DataFrame do pandas, estrutura
que facilita a manipulação, exploração e análise dos dados ao longo do projeto.

In [26]:
linhas = []

for item in dados_reviews["reviews"]:
    appid = item["appid"]
    dados = item["review_data"]

    for review in dados["reviews"]:
        registro = review.copy()
        registro["appid"] = appid
        linhas.append(registro)

df_reviews = pd.DataFrame(linhas)

print(f"DataFrame criado com {len(df_reviews)} linhas e {len(df_reviews.columns)} colunas.")
display(df_reviews.head(3))

DataFrame criado com 37778 linhas e 18 colunas.


,recommendationid,author,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,primarily_steam_deck,appid,timestamp_dev_responded,developer_response
0,201194290,"{'steamid': '76561198152964416', 'num_games_ow...",russian,Я верю что однажды капитализм очиститься и люд...,1754033180,1754033180,True,58,2,0.834045231342315674,2,True,False,False,False,2210,NaN,NaN
1,202221261,"{'steamid': '76561198060071144', 'num_games_ow...",brazilian,Jogo é bom no geral. Mas uma pena não ter se q...,1755311121,1755311121,True,4,0,0.583333313465118408,0,True,False,False,False,2210,NaN,NaN
2,201240332,"{'steamid': '76561197965801053', 'num_games_ow...",english,This game feels like a mix of Quake 2+3 and so...,1754084409,1754084409,True,8,0,0.54731827974319458,0,True,False,False,False,2210,NaN,NaN


## 5. Distribuição das recomendações

A variável `voted_up` indica se o usuário recomendou ou não o jogo. O valor
`True` representa uma recomendação positiva, enquanto `False` representa uma
recomendação negativa.

A análise dessa variável é importante porque ela será utilizada posteriormente
como variável-alvo na tarefa de classificação automática das avaliações.

In [27]:
df_reviews["voted_up"].value_counts()

voted_up
True     28546
False     9232
Name: count, dtype: int64

In [28]:
print("========================================")
print("DISTRIBUIÇÃO DAS RECOMENDAÇÕES")
print("========================================")

print(df_reviews["voted_up"].value_counts())

print("\nPercentual (%):")
display(
    df_reviews["voted_up"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DISTRIBUIÇÃO DAS RECOMENDAÇÕES
voted_up
True     28546
False     9232
Name: count, dtype: int64

Percentual (%):


voted_up
True     75.56
False    24.44
Name: proportion, dtype: float64

## 6. Distribuição dos idiomas

Nesta etapa é analisada a distribuição dos idiomas presentes nas avaliações.
Essa informação é utilizada para definir o recorte linguístico da análise.

Para as etapas de Processamento de Linguagem Natural, serão utilizadas
avaliações em inglês, buscando reduzir a complexidade linguística e manter
maior consistência no processamento dos textos.

In [29]:
print("========================================")
print("IDIOMAS MAIS FREQUENTES")
print("========================================")

display(df_reviews["language"].value_counts().head(15))

IDIOMAS MAIS FREQUENTES


language
english      19487
schinese      5209
russian       3756
german        1441
brazilian     1193
spanish       1073
japanese       932
french         927
koreana        873
turkish        650
tchinese       507
polish         467
latam          252
italian        240
czech          142
Name: count, dtype: int64

## 7. Período das avaliações

O campo `timestamp_created` contém o momento em que cada avaliação foi
publicada. O timestamp Unix é convertido para o formato de data utilizando
a biblioteca pandas.

Essa transformação permite analisar a distribuição temporal das avaliações
e definir o período utilizado no estudo.

In [30]:
df_reviews["data"] = pd.to_datetime(
    df_reviews["timestamp_created"],
    unit="s"
)

print("========================================")
print("PERÍODO DAS AVALIAÇÕES")
print("========================================")
print("Primeira avaliação:", df_reviews["data"].min())
print("Última avaliação:", df_reviews["data"].max())

PERÍODO DAS AVALIAÇÕES
Primeira avaliação: 2010-11-20 20:38:46
Última avaliação: 2025-09-01 06:47:21


## 8. Distribuição das avaliações por ano

Após a conversão das datas, é extraído o ano de publicação de cada avaliação.
A quantidade de avaliações por ano permite observar a disponibilidade dos dados
ao longo do período analisado.

In [31]:
df_reviews["ano"] = df_reviews["data"].dt.year

print("========================================")
print("AVALIAÇÕES POR ANO")
print("========================================")

display(
    df_reviews["ano"]
    .value_counts()
    .sort_index()
)

AVALIAÇÕES POR ANO


ano
2010        5
2011       30
2012       43
2013      113
2014      656
2015      894
2016     1312
2017     2690
2018     2118
2019     2151
2020     2493
2021     2671
2022     2636
2023     2207
2024     2279
2025    15480
Name: count, dtype: int64

## 9. Análise do tamanho dos textos

Nesta etapa é calculada a quantidade de palavras presente em cada avaliação.
Essa análise permite conhecer a extensão dos textos e identificar possíveis
avaliações muito curtas antes da aplicação das técnicas de NLP.

Neste momento, avaliações curtas não são removidas; apenas são identificadas
e quantificadas para apoiar decisões posteriores.

In [32]:
df_reviews["quantidade_palavras"] = (
    df_reviews["review"]
    .fillna("")
    .str.split()
    .str.len()
)

print("========================================")
print("ESTATÍSTICA DO TAMANHO DOS TEXTOS")
print("========================================")

display(df_reviews["quantidade_palavras"].describe())

ESTATÍSTICA DO TAMANHO DOS TEXTOS


count    37778.000000
mean        59.664964
std        116.518534
min          0.000000
25%          5.000000
50%         20.000000
75%         62.000000
max       3458.000000
Name: quantidade_palavras, dtype: float64

## 10. Verificação de valores ausentes

A qualidade dos dados é verificada por meio da identificação de valores
ausentes em cada variável da base.

Essa etapa permite avaliar possíveis limitações dos dados e verificar se
campos importantes para a análise apresentam informações faltantes.

In [33]:
print("========================================")
print("MAPEAMENTO DE VALORES AUSENTES")
print("========================================")

display(
    df_reviews.isnull()
    .sum()
    .sort_values(ascending=False)
)

MAPEAMENTO DE VALORES AUSENTES


developer_response             36867
timestamp_dev_responded        36867
language                           0
author                             0
recommendationid                   0
timestamp_created                  0
review                             0
timestamp_updated                  0
voted_up                           0
weighted_vote_score                0
comment_count                      0
votes_up                           0
votes_funny                        0
received_for_free                  0
steam_purchase                     0
primarily_steam_deck               0
written_during_early_access        0
appid                              0
data                               0
ano                                0
quantidade_palavras                0
dtype: int64

## 11. Recorte por idioma

Para a etapa de Processamento de Linguagem Natural (NLP), serão consideradas
as avaliações escritas em inglês. A utilização de um único idioma reduz a
complexidade do processamento textual e mantém maior consistência linguística
na análise e na classificação das avaliações.

In [34]:
print("========================================")
print("RECORTE POR IDIOMA")
print("========================================")

df_ingles = df_reviews[
    df_reviews["language"] == "english"
].copy()

print("Avaliações em inglês:", len(df_ingles))

RECORTE POR IDIOMA
Avaliações em inglês: 19487


## 12. Distribuição temporal das avaliações em inglês

Após a seleção das avaliações em inglês, analisamos sua distribuição ao longo
dos anos para verificar a disponibilidade de dados no período considerado.

In [35]:
print("========================================")
print("AVALIAÇÕES EM INGLÊS POR ANO")
print("========================================")

display(
    df_ingles["ano"]
    .value_counts()
    .sort_index()
)

AVALIAÇÕES EM INGLÊS POR ANO


ano
2010       5
2011      18
2012      35
2013      90
2014     447
2015     602
2016     794
2017    1459
2018    1088
2019    1118
2020    1343
2021    1339
2022    1446
2023    1221
2024    1256
2025    7226
Name: count, dtype: int64

## 13. Definição do período de análise

O período de análise foi definido entre 2010 e 2025, abrangendo todo o
intervalo temporal disponível na base de avaliações em inglês.

A manutenção do período completo permite aproveitar a maior quantidade
possível de avaliações e observar a evolução das opiniões dos usuários ao
longo do tempo.

In [36]:
print("========================================")
print("RECORTE TEMPORAL")
print("========================================")

df_final = df_ingles[
    df_ingles["ano"].between(2010, 2025)
].copy()

print("Avaliações após o recorte:", len(df_final))
print("Primeiro ano:", df_final["ano"].min())
print("Último ano:", df_final["ano"].max())

RECORTE TEMPORAL
Avaliações após o recorte: 19487
Primeiro ano: 2010
Último ano: 2025


## 14. Distribuição da variável-alvo

A variável `voted_up` representa a recomendação realizada pelo usuário na
Steam. O valor `True` indica uma recomendação positiva, enquanto `False`
indica uma recomendação negativa.

Essa variável será utilizada posteriormente como variável-alvo para os
modelos de classificação.

In [37]:
print("========================================")
print("DISTRIBUIÇÃO DA RECOMENDAÇÃO - BASE FINAL")
print("========================================")

display(
    df_final["voted_up"].value_counts()
)

print("\nPercentual (%):")

display(
    df_final["voted_up"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DISTRIBUIÇÃO DA RECOMENDAÇÃO - BASE FINAL


voted_up
True     14613
False     4874
Name: count, dtype: int64


Percentual (%):


voted_up
True     74.99
False    25.01
Name: proportion, dtype: float64

## 15. Qualidade e tamanho dos textos

Antes da aplicação das técnicas de Processamento de Linguagem Natural,
analisamos o tamanho das avaliações para identificar a distribuição dos
textos e possíveis avaliações excessivamente curtas.

In [38]:
print("========================================")
print("ESTATÍSTICA DO TAMANHO DOS TEXTOS")
print("========================================")

display(
    df_final["quantidade_palavras"].describe()
)

ESTATÍSTICA DO TAMANHO DOS TEXTOS


count    19487.000000
mean        78.963771
std        130.146754
min          0.000000
25%         13.000000
50%         35.000000
75%         88.000000
max       1471.000000
Name: quantidade_palavras, dtype: float64

## 16. Avaliações muito curtas

A quantidade de palavras é analisada para identificar avaliações com pouco
conteúdo textual. Neste momento, essas avaliações não serão removidas,
apenas quantificadas para apoiar decisões posteriores relacionadas ao
processamento de linguagem natural.

In [39]:
print("========================================")
print("TEXTOS MUITO CURTOS - BASE FINAL")
print("========================================")

for limite in [0, 1, 2, 3, 5, 10]:
    quantidade = (
        df_final["quantidade_palavras"] <= limite
    ).sum()

    percentual = quantidade / len(df_final) * 100

    print(
        f"Até {limite} palavras: "
        f"{quantidade} ({percentual:.2f}%)"
    )

TEXTOS MUITO CURTOS - BASE FINAL
Até 0 palavras: 10 (0.05%)
Até 1 palavras: 651 (3.34%)
Até 2 palavras: 1140 (5.85%)
Até 3 palavras: 1579 (8.10%)
Até 5 palavras: 2370 (12.16%)
Até 10 palavras: 4103 (21.06%)


## 17. Verificação de duplicidades

A presença de avaliações duplicadas pode introduzir viés na análise e no
treinamento dos modelos. Por isso, verificamos a existência de avaliações
textualmente duplicadas na base final.

In [40]:
print("========================================")
print("VERIFICAÇÃO DE DUPLICIDADES")
print("========================================")

duplicadas = df_final.duplicated(
    subset=["review"]
).sum()

print("Reviews duplicadas:", duplicadas)

VERIFICAÇÃO DE DUPLICIDADES
Reviews duplicadas: 384


## 18. Resumo da base definitiva

Após as etapas de seleção e validação, consolidamos as principais
características da base que será utilizada nas etapas seguintes do projeto.

In [41]:
print("========================================")
print("RESUMO DA BASE FINAL")
print("========================================")

print("Avaliações:", len(df_final))
print("Período:", df_final["ano"].min(), "a", df_final["ano"].max())
print("Idioma: Inglês")

print(
    "Recomendação positiva:",
    (df_final["voted_up"] == True).sum()
)

print(
    "Recomendação negativa:",
    (df_final["voted_up"] == False).sum()
)

print(
    "Reviews duplicadas:",
    df_final.duplicated(subset=["review"]).sum()
)

RESUMO DA BASE FINAL
Avaliações: 19487
Período: 2010 a 2025
Idioma: Inglês
Recomendação positiva: 14613
Recomendação negativa: 4874
Reviews duplicadas: 384
